In [0]:
-- ============================================================
-- NOTEBOOK: 03_gold_aggregations.sql
-- PURPOSE : Business-level aggregations — Silver → Gold.
-- CATALOG : dev | uat | prod  (passed as :catalog parameter)
-- AUTHOR  : The Data Master | Naval Yemul
-- ============================================================

-- Lakeflow Job SQL Task Parameters:
--   catalog = dev   (or uat / prod)

-- GOLD TABLES:
--   1. customer_360                  — unified customer KPI view
--   2. monthly_transaction_summary   — monthly spend trends
--   3. merchant_category_spend       — spend by category + fraud rate
--   4. loan_portfolio_summary        — loan book health + NPA
--   5. fraud_watchlist               — high-risk flagged customers


-- ─────────────────────────────────────────────────────────────
-- GOLD 1 : customer_360
-- One row per customer — profile + accounts + txns + loans + risk
-- ─────────────────────────────────────────────────────────────

CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.customer_360')
  USING DELTA
  TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'pipeline.layer'                   = 'gold',
    'pipeline.loaded_by'               = 'naval.yemul'
  )
  COMMENT 'Gold: 360° customer view — profile + account + transaction + loan KPIs + risk score.'
AS
WITH account_agg AS (
  SELECT
    customer_id,
    COUNT(DISTINCT account_id)                                AS total_accounts,
    SUM(balance)                                              AS total_balance,
    SUM(credit_limit)                                         AS total_credit_limit,
    SUM(available_credit)                                     AS total_available_credit,
    MAX(last_transaction_date)                                AS latest_account_date,
    COUNT(CASE WHEN status = 'ACTIVE'  THEN 1 END)           AS active_accounts,
    COUNT(CASE WHEN status = 'DORMANT' THEN 1 END)           AS dormant_accounts
  FROM IDENTIFIER(:catalog || '.silver.accounts')
  GROUP BY customer_id
),
txn_agg AS (
  SELECT
    customer_id,
    COUNT(DISTINCT transaction_id)                            AS total_transactions,
    SUM(amount)                                               AS total_txn_amount,
    ROUND(AVG(amount), 2)                                     AS avg_txn_amount,
    MAX(amount)                                               AS max_txn_amount,
    MIN(transaction_date)                                     AS first_txn_date,
    MAX(transaction_date)                                     AS last_txn_date,
    COUNT(CASE WHEN is_flagged = TRUE   THEN 1 END)          AS flagged_txn_count,
    COUNT(CASE WHEN status = 'FAILED'   THEN 1 END)          AS failed_txn_count,
    COUNT(CASE WHEN transaction_type = 'CREDIT' THEN 1 END)  AS credit_count,
    COUNT(CASE WHEN transaction_type = 'DEBIT'  THEN 1 END)  AS debit_count
  FROM IDENTIFIER(:catalog || '.silver.transactions')
  GROUP BY customer_id
),
loan_agg AS (
  SELECT
    customer_id,
    COUNT(DISTINCT loan_id)                                   AS total_loans,
    SUM(principal_amount)                                     AS total_principal,
    SUM(outstanding_balance)                                  AS total_outstanding,
    SUM(emi_amount)                                           AS total_monthly_emi,
    MAX(overdue_days)                                         AS max_overdue_days,
    COUNT(CASE WHEN loan_health = 'DEFAULTED' THEN 1 END)    AS defaulted_loans,
    COUNT(CASE WHEN loan_health = 'AT_RISK'   THEN 1 END)    AS at_risk_loans,
    COUNT(CASE WHEN status = 'ACTIVE'         THEN 1 END)    AS active_loans
  FROM IDENTIFIER(:catalog || '.silver.loans')
  GROUP BY customer_id
)
SELECT
  c.customer_id,
  c.full_name,
  c.email,
  c.country,
  c.account_type,
  c.credit_score,
  c.annual_income_usd,
  c.created_date                                              AS customer_since,
  c.is_active,

  -- account KPIs
  COALESCE(a.total_accounts,         0)                      AS total_accounts,
  COALESCE(a.total_balance,          0.0)                    AS total_balance,
  COALESCE(a.total_credit_limit,     0.0)                    AS total_credit_limit,
  COALESCE(a.total_available_credit, 0.0)                    AS total_available_credit,
  a.active_accounts,
  a.dormant_accounts,

  -- transaction KPIs
  COALESCE(t.total_transactions, 0)                          AS total_transactions,
  COALESCE(t.total_txn_amount,   0.0)                        AS total_txn_amount,
  COALESCE(t.avg_txn_amount,     0.0)                        AS avg_txn_amount,
  COALESCE(t.max_txn_amount,     0.0)                        AS max_txn_amount,
  t.first_txn_date,
  t.last_txn_date,
  COALESCE(t.flagged_txn_count,  0)                          AS flagged_txn_count,
  COALESCE(t.failed_txn_count,   0)                          AS failed_txn_count,
  COALESCE(t.credit_count,       0)                          AS credit_txn_count,
  COALESCE(t.debit_count,        0)                          AS debit_txn_count,

  -- loan KPIs
  COALESCE(l.total_loans,        0)                          AS total_loans,
  COALESCE(l.total_principal,    0.0)                        AS total_loan_principal,
  COALESCE(l.total_outstanding,  0.0)                        AS total_outstanding_balance,
  COALESCE(l.total_monthly_emi,  0.0)                        AS total_monthly_emi,
  COALESCE(l.max_overdue_days,   0)                          AS max_overdue_days,
  COALESCE(l.defaulted_loans,    0)                          AS defaulted_loans,
  COALESCE(l.at_risk_loans,      0)                          AS at_risk_loans,

  -- composite risk score 0–100
  LEAST(100, ROUND(
      COALESCE(t.flagged_txn_count, 0) * 5.0
    + COALESCE(l.defaulted_loans,   0) * 20.0
    + COALESCE(l.at_risk_loans,     0) * 10.0
    + CASE WHEN COALESCE(l.max_overdue_days, 0) > 90 THEN 15 ELSE 0 END
    + CASE WHEN c.credit_score < 500                  THEN 10 ELSE 0 END
  , 2))                                                       AS risk_score,

  CASE
    WHEN LEAST(100,
        COALESCE(t.flagged_txn_count, 0) * 5.0
      + COALESCE(l.defaulted_loans,   0) * 20.0
      + COALESCE(l.at_risk_loans,     0) * 10.0
      + CASE WHEN COALESCE(l.max_overdue_days, 0) > 90 THEN 15 ELSE 0 END
      + CASE WHEN c.credit_score < 500                  THEN 10 ELSE 0 END
    ) >= 50 THEN 'HIGH'
    WHEN LEAST(100,
        COALESCE(t.flagged_txn_count, 0) * 5.0
      + COALESCE(l.defaulted_loans,   0) * 20.0
      + COALESCE(l.at_risk_loans,     0) * 10.0
      + CASE WHEN COALESCE(l.max_overdue_days, 0) > 90 THEN 15 ELSE 0 END
      + CASE WHEN c.credit_score < 500                  THEN 10 ELSE 0 END
    ) >= 20 THEN 'MEDIUM'
    ELSE 'LOW'
  END                                                         AS risk_tier,

  current_timestamp()                                         AS _gold_loaded_at,
  :catalog                                                    AS _catalog
FROM IDENTIFIER(:catalog || '.silver.customers') c
LEFT JOIN account_agg a ON c.customer_id = a.customer_id
LEFT JOIN txn_agg     t ON c.customer_id = t.customer_id
LEFT JOIN loan_agg    l ON c.customer_id = l.customer_id;


-- ─────────────────────────────────────────────────────────────
-- GOLD 2 : monthly_transaction_summary
-- ─────────────────────────────────────────────────────────────

CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.monthly_transaction_summary')
  USING DELTA
  PARTITIONED BY (transaction_year, transaction_month)
  TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'pipeline.layer'                   = 'gold',
    'pipeline.loaded_by'               = 'naval.yemul'
  )
  COMMENT 'Gold: monthly spend trends per customer per currency.'
AS
SELECT
  t.customer_id,
  c.full_name,
  c.country,
  t.currency,
  t.transaction_year,
  t.transaction_month,
  COUNT(DISTINCT t.transaction_id)                                      AS txn_count,
  ROUND(SUM(t.amount), 2)                                               AS total_amount,
  ROUND(SUM(CASE WHEN t.transaction_type = 'DEBIT'  THEN t.amount ELSE 0 END), 2) AS total_debit,
  ROUND(SUM(CASE WHEN t.transaction_type = 'CREDIT' THEN t.amount ELSE 0 END), 2) AS total_credit,
  ROUND(AVG(t.amount), 2)                                               AS avg_amount,
  ROUND(MAX(t.amount), 2)                                               AS max_single_txn,
  COUNT(CASE WHEN t.is_flagged = TRUE  THEN 1 END)                      AS flagged_count,
  COUNT(CASE WHEN t.status = 'FAILED'  THEN 1 END)                      AS failed_count,
  current_timestamp()                                                   AS _gold_loaded_at,
  :catalog                                                              AS _catalog
FROM IDENTIFIER(:catalog || '.silver.transactions') t
LEFT JOIN IDENTIFIER(:catalog || '.silver.customers') c
  ON t.customer_id = c.customer_id
GROUP BY
  t.customer_id, c.full_name, c.country,
  t.currency, t.transaction_year, t.transaction_month;


-- ─────────────────────────────────────────────────────────────
-- GOLD 3 : merchant_category_spend
-- ─────────────────────────────────────────────────────────────

CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.merchant_category_spend')
  USING DELTA
  TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'pipeline.layer'                   = 'gold',
    'pipeline.loaded_by'               = 'naval.yemul'
  )
  COMMENT 'Gold: spend by merchant category + channel with fraud rate.'
AS
SELECT
  merchant_category,
  channel,
  currency,
  COUNT(DISTINCT transaction_id)                              AS txn_count,
  COUNT(DISTINCT customer_id)                                 AS unique_customers,
  ROUND(SUM(amount),  2)                                      AS total_spend,
  ROUND(AVG(amount),  2)                                      AS avg_spend,
  ROUND(MAX(amount),  2)                                      AS max_spend,
  COUNT(CASE WHEN is_flagged = TRUE THEN 1 END)               AS flagged_count,
  ROUND(
    COUNT(CASE WHEN is_flagged = TRUE THEN 1 END) * 100.0
    / NULLIF(COUNT(*), 0), 2
  )                                                           AS fraud_rate_pct,
  current_timestamp()                                         AS _gold_loaded_at,
  :catalog                                                    AS _catalog
FROM IDENTIFIER(:catalog || '.silver.transactions')
GROUP BY merchant_category, channel, currency
ORDER BY total_spend DESC;


-- ─────────────────────────────────────────────────────────────
-- GOLD 4 : loan_portfolio_summary
-- ─────────────────────────────────────────────────────────────

CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.loan_portfolio_summary')
  USING DELTA
  TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'pipeline.layer'                   = 'gold',
    'pipeline.loaded_by'               = 'naval.yemul'
  )
  COMMENT 'Gold: loan book KPIs by type, health bucket and collateral. Includes NPA rate.'
AS
SELECT
  loan_type,
  loan_health,
  status,
  collateral_type,
  COUNT(DISTINCT loan_id)                                     AS loan_count,
  COUNT(DISTINCT customer_id)                                 AS unique_borrowers,
  ROUND(SUM(principal_amount),    2)                          AS total_principal,
  ROUND(SUM(outstanding_balance), 2)                          AS total_outstanding,
  ROUND(SUM(emi_amount),          2)                          AS total_monthly_emi,
  ROUND(AVG(interest_rate_pct),   3)                          AS avg_interest_rate,
  ROUND(AVG(repayment_pct),       2)                          AS avg_repayment_pct,
  MAX(overdue_days)                                           AS max_overdue_days,
  ROUND(AVG(overdue_days),        1)                          AS avg_overdue_days,
  ROUND(SUM(
    CASE WHEN overdue_days > 90 THEN outstanding_balance ELSE 0 END
  ), 2)                                                       AS npa_outstanding,
  ROUND(
    SUM(CASE WHEN overdue_days > 90 THEN outstanding_balance ELSE 0 END) * 100.0
    / NULLIF(SUM(outstanding_balance), 0), 2
  )                                                           AS npa_rate_pct,
  current_timestamp()                                         AS _gold_loaded_at,
  :catalog                                                    AS _catalog
FROM IDENTIFIER(:catalog || '.silver.loans')
GROUP BY loan_type, loan_health, status, collateral_type
ORDER BY total_outstanding DESC;


-- ─────────────────────────────────────────────────────────────
-- GOLD 5 : fraud_watchlist
-- ─────────────────────────────────────────────────────────────

CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.fraud_watchlist')
  USING DELTA
  TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'pipeline.layer'                   = 'gold',
    'pipeline.loaded_by'               = 'naval.yemul'
  )
  COMMENT 'Gold: customers with fraud signals — feeds ops alert dashboard.'
AS
WITH flagged AS (
  SELECT
    customer_id,
    COUNT(DISTINCT transaction_id)                                        AS total_txns,
    COUNT(CASE WHEN is_flagged = TRUE  THEN 1 END)                        AS flagged_txns,
    COUNT(CASE WHEN status     = 'FAILED' THEN 1 END)                     AS failed_txns,
    ROUND(SUM(CASE WHEN is_flagged = TRUE THEN amount ELSE 0 END), 2)     AS flagged_amount,
    MAX(transaction_date)                                                  AS last_flagged_date,
    COLLECT_SET(merchant_category)                                         AS flagged_categories
  FROM IDENTIFIER(:catalog || '.silver.transactions')
  GROUP BY customer_id
  HAVING COUNT(CASE WHEN is_flagged = TRUE THEN 1 END) > 0
)
SELECT
  f.customer_id,
  c.full_name,
  c.email,
  c.country,
  c.credit_score,
  f.total_txns,
  f.flagged_txns,
  f.failed_txns,
  f.flagged_amount,
  ROUND(f.flagged_txns * 100.0 / NULLIF(f.total_txns, 0), 2)            AS fraud_rate_pct,
  f.last_flagged_date,
  f.flagged_categories,
  CASE
    WHEN f.flagged_txns >= 5 OR f.flagged_amount > 100000               THEN 'CRITICAL'
    WHEN f.flagged_txns >= 3                                             THEN 'HIGH'
    ELSE                                                                      'MONITOR'
  END                                                                     AS alert_tier,
  current_timestamp()                                                     AS _gold_loaded_at,
  :catalog                                                                AS _catalog
FROM flagged f
JOIN IDENTIFIER(:catalog || '.silver.customers') c
  ON f.customer_id = c.customer_id
ORDER BY f.flagged_txns DESC, f.flagged_amount DESC;


-- ─────────────────────────────────────────────────────────────
-- GOLD VALIDATION
-- ─────────────────────────────────────────────────────────────

SELECT 'gold.customer_360'             AS table_name, COUNT(*) AS rows
FROM IDENTIFIER(:catalog || '.gold.customer_360')
UNION ALL
SELECT 'gold.monthly_transaction_summary', COUNT(*)
FROM IDENTIFIER(:catalog || '.gold.monthly_transaction_summary')
UNION ALL
SELECT 'gold.merchant_category_spend', COUNT(*)
FROM IDENTIFIER(:catalog || '.gold.merchant_category_spend')
UNION ALL
SELECT 'gold.loan_portfolio_summary',  COUNT(*)
FROM IDENTIFIER(:catalog || '.gold.loan_portfolio_summary')
UNION ALL
SELECT 'gold.fraud_watchlist',         COUNT(*)
FROM IDENTIFIER(:catalog || '.gold.fraud_watchlist');